# Introduction to TauREx 3

We will use this notebook to introduce ourselves to the TauREx platform. For this introduction we will go through the basics of:

- Setting up the notebook environment
- Acquiring some opacity data
- Creating a model
- Running the model

First, set up the notebook so it can import the local TauREx source tree and store downloaded data in a temporary folder that is ignored by git:

In [1]:
from pathlib import Path
import sys
from urllib.request import urlretrieve


def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "taurex").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the TauREx project root from the current notebook directory."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
TMP_DIR = PROJECT_ROOT / "examples" / "tmp"
XSEC_DIR = TMP_DIR / "xsec"
CIA_DIR = TMP_DIR / "cia"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

XSEC_DIR.mkdir(parents=True, exist_ok=True)
CIA_DIR.mkdir(parents=True, exist_ok=True)


def download_file(url: str, destination: Path) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        print(f"Skipping {destination.name}; file already exists.")
        return destination
    print(f"Downloading {destination.name}...")
    urlretrieve(url, destination)
    return destination


print(f"Using TauREx source: {SRC_DIR}")
print(f"Temporary data directory: {TMP_DIR}")
print(f"Opacity data directory: {XSEC_DIR}")
print(f"CIA data directory: {CIA_DIR}")

Using TauREx source: /home/simone/Dropbox/eScience_projects/TauRex/taurex3/src
Temporary data directory: /home/simone/Dropbox/eScience_projects/TauRex/taurex3/examples/tmp
Opacity data directory: /home/simone/Dropbox/eScience_projects/TauRex/taurex3/examples/tmp/xsec
CIA data directory: /home/simone/Dropbox/eScience_projects/TauRex/taurex3/examples/tmp/cia


Checking its version it a good way to see if its installed:

In [ ]:
import taurex
taurex.__version__

If you have any warnings don't worry, these are optional features that can be enabled by installing extra libraries.

Now before we move on we should get some opacity data.

# Opacities

Opacities describe optical behaviour of some physical property in the atmosphere. Some can be computed while others must be loaded in.
The main opacities used in TauREx are molecular absorption cross-sections.

These require computation from a molecular line-list such as those from the [ExoMol](https://www.exomol.com) project but thankfully these generous people have computed some already for use.

TauREx supports many formats


For our purposes we will be using 4 molecules:

- $H_2O$ from [POZAKATEL linelist](https://exomol.com/data/molecules/H2O/1H2-16O/POKAZATEL/)
- $CH_4$ from [MM linelist](https://exomol.com/data/molecules/CH4/12C-1H4/MM/)
- $NH_3$ from [CoYuTe linelist](https://exomol.com/data/molecules/NH3/14N-1H3/CoYuTe/)
- $CO_2$ from [UCL-4000 linelist](https://exomol.com/data/molecules/CO2/12C-16O2/UCL-4000/)

*Please make sure you site the linelists individually if you use them for your work*

In [ ]:
EXOMOL_URLS = {
    "H2O": "https://exomol.com/db/H2O/1H2-16O/POKAZATEL/1H2-16O__POKAZATEL__R15000_0.3-50mu.xsec.TauREx.h5",
    "CO2": "https://exomol.com/db/CO2/12C-16O2/UCL-4000/12C-16O2__UCL-4000.R15000_0.3-50mu.xsec.TauREx.h5",
    "CH4": "https://exomol.com/db/CH4/12C-1H4/MM/12C-1H4__MM.R15000_0.3-50mu.xsec.TauREx.h5",
    "NH3": "https://exomol.com/db/NH3/14N-1H3/CoYuTe/14N-1H3__CoYuTe.R15000_0.3-50mu.xsec.TauREx.h5",
}

for molecule, url in EXOMOL_URLS.items():
    filename = url.rsplit("/", 1)[-1]
    download_file(url, XSEC_DIR / filename)

Additionally we will also like to include conditionally induced absorption. We will grab these from the [HITRAN](https://hitran.org/) database. Specifically $H_2-H_2$ and $H_2-He$ pairs.

In [ ]:
CIA_URLS = {
    "H2-H2_eq_2018.cia": "https://hitran.org/data/CIA/alternate/H2-H2_eq_2018.cia",
    "H2-He_eq_2011.cia": "https://hitran.org/data/CIA/alternate/H2-He_eq_2011.cia",
}

for filename, url in CIA_URLS.items():
    download_file(url, CIA_DIR / filename)

# Loading in our opacities

TauREx makes use of an automated opacity loader. By pointing to a directory, TauREx will attempt to discover, determine the filetype and identify the molecule. For molecular cross-sections, TauREx supports:

- Legacy TauREx 2 ``pickle`` format
- [ExomolOP](https://www.aanda.org/articles/aa/full_html/2021/02/aa38350-20/aa38350-20.html) HDF5 format.
- [ExoTransmit](https://github.com/elizakempton/Exo_Transmit) format

Additionally, some ``plugins`` can extend the formats that can be used.

For CIA we support:

- Legacy TauREx 2 ``.db`` format
- HITRAN ``.cia`` format

To load in our opacities we can make use of the ``OpacityCache`` and ``CIACache`` objects and point them to our folders.

In [ ]:
from taurex.cache import CIACache, OpacityCache

OpacityCache().set_opacity_path(str(XSEC_DIR))
CIACache().set_cia_path(str(CIA_DIR))

🚀 **TIP**
``OpacityCache`` and ``CIACache`` are actually [Singletons](https://en.wikipedia.org/wiki/Singleton_pattern). This means that calling their constructor will always return the same object!

Now we can see what the ``OpacityCache`` has found:

In [ ]:
OpacityCache().find_list_of_molecules()

Our molecules were found! Great! Now we can move on to building our model

# Building models

TauREx constructs a simulation by putting together pieces to form a complete model.

The baseline TauREx can build three types of models:

- [TransmissionModel](https://taurex3.readthedocs.io/en/latest/api/taurex.model.html#taurex.model.transmission.TransmissionModel) - Transit spectral modelling
- [EmissionModel](https://taurex3.readthedocs.io/en/latest/api/taurex.model.html#taurex.model.emission.EmissionModel) - Eclipse spectral modelling
- [DirectImageModel](https://taurex3.readthedocs.io/en/latest/api/taurex.model.html#taurex.model.directimage.DirectImageModel) - Direct Imaging model

In order to accomplish this, we need to build some of the pieces first:

1. Temperature Model
2. Pressure Model
3. Chemical Model
4. Optical depth Model.
5. Planet
6. Star


## Temperature model

Vanilla TauREx 3 has a few temperature models available:

- ``Isothermal``
- [Guillot2010](https://www.aanda.org/articles/aa/abs/2010/12/aa13396-09/aa13396-09.html)
- ``NPoint``
- ``TemperatureFile``

All of them exist under the ``taurex.temperature`` module. Lets load up an ``Isothermal`` model and see how we create a profile that is a constant $2000~K$ along the atmosphere.

In [ ]:
from taurex.temperature import Isothermal

iso_t = Isothermal(T=2000.0)

Easy! Other profiles can be created the same way. You can always ask for help if you want to understand how to build one of them:

In [ ]:
from taurex.temperature import Guillot2010

Guillot2010?

## Pressure Profiles

Currently TauREx supports 2 types of pressure profiles:

- ``SimplePressureProfile``
- ``FilePressure``

The ``SimplePressureProfile`` profile simply builds a pressure profile equally spaced in the log scale.

Units are in $Pa$

In [ ]:
from taurex.pressure import SimplePressureProfile

press = SimplePressureProfile(
    nlayers=100,
    atm_min_pressure=1e-5,
    atm_max_pressure=1e5
)


## Chemistry Model

Baseline TauREx only has two chemical models:

- ``TaurexChemistry``
- ``ChemistryFile``

In particular ``TaurexChemistry`` is a model that allows for the free determination of chemical profiles for our chosen molecules.

We define **fill gases** which will be used to fill up the remaining atmosphere after we include our molecules.

The ratio is used if we have more that one fill gas.

In [ ]:
from taurex.chemistry import TaurexChemistry

chemistry = TaurexChemistry(fill_gases=["H2", "He"], ratio=0.1756)

### Adding molecules

Now we can start adding molecules, ``TaurexChemistry`` has the ``addGas`` method which can be used to insert molecules into the chemical model. We can use the ``ConstantGas`` profile but there is the ``TwoLayerGas`` available as well.

TauREx uses *volume mixing ratio (VMR)*

In [ ]:
from taurex.chemistry import ConstantGas

gas = ConstantGas(molecule_name="H2O", mix_ratio=1e-3)
chemistry.addGas(gas)

We can also create it on the spot:

In [ ]:
chemistry.addGas(ConstantGas(molecule_name="CH4", mix_ratio=1e-4))

And also chain ``addGas`` calls:

In [ ]:
chemistry.addGas(ConstantGas(molecule_name="NH3", mix_ratio=1e-4)).addGas(ConstantGas(molecule_name="CO2", mix_ratio=1e-4))
#

Now chemistry has an interesting property. There is an ``activeGases`` and ``inactiveGases``

*Active Gases:*
- Have molecular cross-sections loaded in

*Inactive Gases:*
- Do not have molecular cross-sections

Now inactive does not mean they are completely spectrscopically inactive (as they may come in the form of CIA or rayleigh).

We can check them with these properties:

In [ ]:
chemistry.inactiveGases, chemistry.activeGases

Our chemistry is setup!

## Star and Planet

TauRex requires information about the planet and star in the system in order to simulate the spectra.
The planet contains properties such as:
- Mass ($M_{jup}$)
- Radius ($R_{jup}$)
- Semi-Major Axis ($AU$)
- Impact parameter
- Orbital Period ($day$)
- Albedo
- Transit time ($s$)

Not all of these are required for a particular spectrum. For example, transits only require ``mass`` and ``radius`` to be defined.

Lets create our favourite planet HD 209458 b

In [ ]:
from taurex.planet import Planet

planet = Planet(planet_mass=0.74, planet_radius=1.38)


For the star we can choose between:

- ``BlackbodyStar``
- ``PhoenixStar``

For this purpose we will stick with the ``BlackbodyStar`` as we do not possess the pre-requisite PHOENIX spectrum yet.

With the star we can define these properties:

- Effective Temperaure ($K$)
- Radius ($R_{sun}$)
- Distance ($pc$)
- Mass ($M_{sun}$)
- metallicity ($Z_{sun}$)

Again for our purposes, only ``temperature`` and ``radius`` are needed for transits and eclipse. Direct image requires the ``distance`` parameter.

In [ ]:
from taurex.stellar import BlackbodyStar

star = BlackbodyStar(temperature=6117, radius=1.16)